In [2]:
import torch 
import torch.nn as nn
import torch.optim as optim

In [3]:
import pandas as pd
import numpy as np

In [4]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

In [5]:
from datasets import load_dataset
dataset = load_dataset("parquet", data_files={'train': 'stanford_cars/data/train-*.parquet'})
print(dataset['train'])

c:\Users\prana\anaconda3\envs\dl\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['image', 'label'],
    num_rows: 8144
})


In [6]:
test_dataset = load_dataset("parquet", data_files={'test': 'stanford_cars/data/train-*.parquet'})
print(test_dataset['test'])

Dataset({
    features: ['image', 'label'],
    num_rows: 8144
})


In [6]:
df = pd.DataFrame(dataset['train'])

In [7]:
df2 = pd.DataFrame(test_dataset['test'])

In [10]:
df['label'].value_counts()

label
118    68
78     49
160    48
166    48
143    47
       ..
174    31
63     30
157    29
98     28
135    24
Name: count, Length: 196, dtype: int64

In [25]:
import io
import torch
from torchvision import transforms
from PIL import Image

# Data Loader and Preprocessing

In [26]:
from torch.utils.data import Dataset, DataLoader

In [27]:
class datasetTrain(Dataset):
    def __init__(self,dataframe):
        
        #cleaning
        self.data = []
        for _, row in dataframe.iterrows():
            img, label = row['image'], row['label']
            if img.width >= 20 and img.height >= 20:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                self.data.append((img, label))

        #preprocessing
        self.preprocess = transforms.Compose([
            # transforms.Resize((224, 224)), # resize the image to 224x224
            # transforms.RandomResizedCrop(size=128,scale=(0.80,1.0),ratio=(0.75,1.33)), # random crop the image to 128, with a crop window of 80% to 100% of the original image, and an aspect ratio of image between 0.75 and 1.33 before resizing
            # transforms.RandomHorizontalFlip(), # Augmentations
            # transforms.ColorJitter(brightness=0.2, contrast=0.2), # Augmentations
            transforms.Resize((128, 128)), # resize the image to 224x224
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalize the image with mean and std of ImageNet dataset
        ])

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img, label = self.data[idx]
        return self.preprocess(img), torch.tensor(label)

In [28]:
class datasetTest(Dataset):
    def __init__(self,dataframe):
        
        #cleaning
        self.data = []
        for _, row in dataframe.iterrows():
            img, label = row['image'], row['label']
            if img.width >= 20 and img.height >= 20:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                self.data.append((img, label))

        #preprocessing
        self.preprocess = transforms.Compose([
            transforms.Resize((128, 128)), # resize the image to 128x128
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalize the image with mean and std of ImageNet dataset
        ])

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img, label = self.data[idx]
        return self.preprocess(img), torch.tensor(label)

# Neural Network

In [29]:
# structuring the ANN: 

class neuralNetwork(nn.Module):
    def __init__(self, num_inputs):
        super().__init__()
        self.flatten = nn.Flatten()
        self.network = nn.Sequential(
            nn.Linear(num_inputs, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 196),
        )
    
    def forward(self, x):
        x = self.flatten(x) # flatten the 3 channel, width, height image into a 1D vector
        y_hat = self.network(x)
        return y_hat

In [30]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = neuralNetwork(128*128*3).to(device) # pass the shape in this 
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(lr=0.0001,params=model.parameters())

# Epochs and Training

In [ ]:
train_loader = DataLoader(datasetTrain(df), batch_size=32, shuffle=True)


In [31]:
test_loader = DataLoader(datasetTest(df2), batch_size=32, shuffle=False)

In [42]:
epochs = 10

for epoch in range(epochs):
    
    model.train()
    total_loss_per_epoch = 0
    for batch_features,batch_labels in train_loader:

        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        #Forward pass
        y_hat = model(batch_features)
        loss = loss_fn(y_hat, batch_labels)

        #backward pass
        optimizer.zero_grad()
        loss.backward()

        #update wieghts
        optimizer.step()
        total_loss_per_epoch += loss.item()
    
    avg_loss = total_loss_per_epoch / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
        

Epoch 1/10, Loss: 1.3155
Epoch 2/10, Loss: 0.8935
Epoch 3/10, Loss: 0.6110
Epoch 4/10, Loss: 0.3889
Epoch 5/10, Loss: 0.2492
Epoch 6/10, Loss: 0.1909
Epoch 7/10, Loss: 0.1294
Epoch 8/10, Loss: 0.1063
Epoch 9/10, Loss: 0.0904
Epoch 10/10, Loss: 0.0746


after 20 epochs its loss is 0.0746

In [43]:
torch.save(model.state_dict(), "stanford_cars_model_ann.pth")

# Evaluation

In [44]:
model.eval()
total=0
correct =0 
with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        
        # Get predictions
        outputs = model(batch_features)
        _, predicted = torch.max(outputs, dim=1)
        
        # Count accuracy
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
print("Test accuracy:", correct / total)

Test accuracy: 0.9947200392927309


In [55]:
test_dataset['test'][123]

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=630x394>,
 'label': 100}

In [62]:
model.load_state_dict(torch.load('stanford_cars_model_ann.pth'))
model.eval()


neuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=49152, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=256, bias=True)
    (5): ReLU()
    (6): Linear(in_features=256, out_features=256, bias=True)
    (7): ReLU()
    (8): Linear(in_features=256, out_features=196, bias=True)
  )
)

In [74]:
# 1. Get the image and its REAL label from the dataset
image, actual_label = test_loader.dataset[72]

# 2. Predict
model.eval()
with torch.no_grad():
    output = model(image.unsqueeze(0).to(device))
    _, pred = torch.max(output, 1)

print(f"Prediction: {pred.item()}")
print(f"Actual Label from Dataset: {actual_label}")

if pred.item() == actual_label:
    print("The model is actually correct for this index!")
else:
    print("There is a genuine mismatch for this specific image.")

Prediction: 1
Actual Label from Dataset: 1
The model is actually correct for this index!
